## Testing our data Pipelines

In [1]:
import pandas as pd
from pathlib import Path

DATA = Path("business-and-market-potential/data")

companies = pd.read_csv(
    DATA / "nj_formd_companies.csv",
    dtype={"cik": str, "zip": str},
    parse_dates=["first_raise", "last_raise"],
)
offerings = pd.read_csv(
    DATA / "nj_formd_offerings.csv",
    dtype={"cik": str, "zip": str, "file_num": str, "accession": str},
    parse_dates=["first_sale_date", "filing_date"],
)

print(companies.shape, offerings.shape)
companies.head()


(778, 19) (1171, 22)


,company,cik,city,zip,industry,incorporated,young_company,revenue_range,num_offerings,total_raised,largest_round,raised_last_24mo,total_investors,first_raise,last_raise,months_since_last_raise,avg_months_between_raises,related_people,edgar_url
0,"CoreWeave, Inc.",1769628,Roseland,07068,Other Technology,overFiveYears,False,Decline to Disclose,3,1.158182e+09,1.149999e+09,0.0,82,2020-12-14,2024-05-16,28.3,20.5,Brannin McBee (Executive Officer); Michael Int...,https://www.sec.gov/cgi-bin/browse-edgar?actio...
1,Attentive Mobile Inc.,1730164,Hoboken,07030,Other Technology,2016,True,Decline to Disclose,2,5.290467e+08,4.699999e+08,0.0,74,2021-03-09,2021-06-09,63.5,3.0,Patrick Grady (Director); Scott Friend (Direct...,https://www.sec.gov/cgi-bin/browse-edgar?actio...
2,"CRB Group, Inc.",1688431,Fort Lee,07024,Other Banking and Financial Services,overFiveYears,False,Decline to Disclose,2,3.501201e+08,3.441201e+08,0.0,91,2021-02-01,2022-01-28,55.9,11.9,David Nachbar (Executive Officer); K. Waterman...,https://www.sec.gov/cgi-bin/browse-edgar?actio...
3,"Monad Labs, Inc.",2022178,Califon,07830,Other Technology,2022,True,Decline to Disclose,3,2.583938e+08,2.162463e+08,0.0,63,2022-06-07,2024-03-12,30.5,10.5,James Hunsaker (Director); Keone Hon (Executiv...,https://www.sec.gov/cgi-bin/browse-edgar?actio...
4,Rhodenergy An Environmental Co Corp,2105633,Princeton,08540,Biotechnology,2024,True,"$5,000,001 - $25,000,000",1,2.500000e+08,2.500000e+08,250000000.0,0,2025-12-31,2025-12-31,8.8,NaN,Irfan Jameel (Executive Officer),https://www.sec.gov/cgi-bin/browse-edgar?actio...


---
## EDA: column fill rate, typical values, nulls/zeros, revenue disclosure

In [2]:
def fill_summary(df):
    n = len(df)
    return pd.DataFrame({
        'dtype': df.dtypes,
        'non_null': df.count(),
        'pct_filled': (df.count() / n * 100).round(1),
        'n_unique': df.nunique(),
    })

print('companies.csv')
display(fill_summary(companies))
print('offerings.csv')
display(fill_summary(offerings))

companies.csv


,dtype,non_null,pct_filled,n_unique
company,str,778,100.0,778
cik,str,778,100.0,778
city,str,778,100.0,233
zip,str,778,100.0,239
industry,str,778,100.0,20
incorporated,str,778,100.0,12
young_company,bool,778,100.0,2
revenue_range,str,778,100.0,8
num_offerings,int64,778,100.0,11
total_raised,float64,778,100.0,539


offerings.csv


,dtype,non_null,pct_filled,n_unique
company,str,1171,100.0,792
cik,str,1171,100.0,778
city,str,1171,100.0,239
zip,str,1171,100.0,243
entity_type,str,1171,100.0,4
incorporated,str,1171,100.0,12
young_company,bool,1171,100.0,2
industry,str,1171,100.0,20
revenue_range,str,1171,100.0,8
first_sale_date,datetime64[us],1070,91.4,755


Every column is technically non-null in both files, **except** `avg_months_between_raises`
(only 26.1% — NaN whenever a company has raised just once, which is expected) and
`first_sale_date` / `total_offering` in offerings.csv (~91%). The real gaps are hidden
inside populated numeric columns as placeholder zeros, and inside `revenue_range` as a
"Decline to Disclose" category — checked below.

In [3]:
# zeros standing in for "no data", not a real zero-dollar value
for col in ['total_raised', 'largest_round', 'raised_last_24mo', 'total_investors', 'num_offerings']:
    zeros = (companies[col] == 0).sum()
    print(f'{col:28s} zero={zeros:4d} ({zeros/len(companies)*100:4.1f}%)  '
          f'nonzero={len(companies)-zeros:4d}  min={companies[col].min()}  '
          f'median={companies[col].median()}  max={companies[col].max()}')

total_raised                 zero=  84 (10.8%)  nonzero= 694  min=0.0  median=1997500.0  max=1158182236.0
largest_round                zero=  84 (10.8%)  nonzero= 694  min=0.0  median=1640000.0  max=1149999332.0
raised_last_24mo             zero= 603 (77.5%)  nonzero= 175  min=0.0  median=0.0  max=250000000.0
total_investors              zero=  74 ( 9.5%)  nonzero= 704  min=0  median=7.0  max=2454
num_offerings                zero=   0 ( 0.0%)  nonzero= 778  min=1  median=1.0  max=38


- `total_raised` / `largest_round`: ~11% are 0 -> undisclosed, not "raised $0"
- `raised_last_24mo`: **77.5% are 0** -> most companies here have not raised in the last 24 months
- `total_investors`: ~9.5% are 0
- `num_offerings` is never 0 (every row filed at least one Form D by definition)

In [4]:
print('total_raised, nonzero rows only:')
display(companies.loc[companies.total_raised > 0, 'total_raised'].describe().apply(lambda x: f'{x:,.0f}'))

total_raised, nonzero rows only:


count              694
mean        14,874,096
std         56,566,883
min                940
25%            500,000
50%          2,635,093
75%         10,000,000
max      1,158,182,236
Name: total_raised, dtype: str

Median disclosed raise (excluding zeros): **~$2.6M**. Mean (**~$14.9M**) is dragged
way up by outliers (CoreWeave $1.16B, Attentive Mobile $529M) — use median, not mean, for
"typical" company.

In [5]:
display(companies.revenue_range.value_counts())
undisclosed = companies.revenue_range.isin(['Decline to Disclose', 'No Revenues'])
print(f"Decline to Disclose + No Revenues = {undisclosed.sum()} / {len(companies)} ({undisclosed.mean()*100:.1f}%)")

revenue_range
Decline to Disclose           629
No Revenues                    82
$1 - $1,000,000                42
$1,000,001 - $5,000,000        12
$5,000,001 - $25,000,000        6
Over $100,000,000               3
Not Applicable                  3
$25,000,001 - $100,000,000      1
Name: count, dtype: int64

Decline to Disclose + No Revenues = 711 / 778 (91.4%)


**~91% of companies give nothing usable on revenue** (81% decline to disclose, 10.5% report
no revenue). Only ~9% (68 companies) report an actual band, and just 3 are over $100M.
`revenue_range` can't support an aggregate "financial health" claim — usable only as a
per-company footnote when present.

In [6]:
print('industry (top 10 of', companies.industry.nunique(), 'total):')
display(companies.industry.value_counts().head(10))
print('young_company:')
display(companies.young_company.value_counts())
print('offerings.csv exemptions claimed:')
display(offerings.exemptions.value_counts())

industry (top 10 of 20 total):


industry
Other Technology                        253
Other                                   229
Other Health Care                        80
Biotechnology                            56
Retailing                                29
Pharmaceuticals                          26
Other Banking and Financial Services     22
Business Services                        15
Construction                             15
Manufacturing                            12
Name: count, dtype: int64

young_company:


young_company
True     567
False    211
Name: count, dtype: int64

offerings.csv exemptions claimed:


exemptions
06b              1036
06c                58
06b, 3C, 3C.1      45
04                 12
06b, 4a5           10
04, 06b             2
04.2                2
4a5                 1
04, 06b, 4a5        1
06b, 3C, 3C.7       1
06b, 3C.1           1
04, 06c, 4a5        1
04.1                1
Name: count, dtype: int64

## Summary: what this dataset can/can't answer

| Question | Answerable? | Notes |
|---|---|---|
| How much raised, total | Yes | `total_raised`, treat 0 as undisclosed, use median not mean |
| Funding frequency / timing | Yes | `num_offerings`, `avg_months_between_raises` (26% of rows have 2+ raises) |
| Raised recently (last 24mo) | Partial | Column exists but 77.5% are 0 |
| Number of investors | Yes | count only, via `total_investors` / `num_investors` |
| Investor identity (VC/fund names) | No | Form D never discloses investor names |
| Revenue / financial performance | Mostly no | ~91% decline to disclose or report no revenue |
| Industry / entity type / exemption type | Yes | fully populated, clean categoricals |

---
## Investor identity for top companies (press-sourced, NOT from Form D)

Form D can't tell us who invested (see above), so for the **top 15 NJ companies by
`total_raised`** we looked up press coverage / funding-round announcements instead. This
is manually researched from news articles and company press releases, not SEC data —
treat it as directional, not verified/complete, and re-check before citing externally.

Four of the top 15 (`HVG Holdings, LLC`, `Estee Capital LLC`, `Alexander Capital Ventures
LLC`, `McN Investments Ltd`) look like investment vehicles / funds / broker-dealers
themselves (entity names, high offering counts e.g. Alexander Capital Ventures = 38
offerings) rather than operating startups raising VC — excluded from the table below since
"who invested in a fund" isn't the same question as "who backed this startup."

In [7]:
investor_research = pd.DataFrame([
    dict(company="CoreWeave, Inc.", form_d_total_raised=1_158_182_236,
         round="Series C ($1.1B, May 2024)",
         lead_investors="Coatue",
         other_investors="Magnetar, Altimeter Capital, Fidelity Management & Research, Lykos Global Management",
         source="prnewswire.com/news-releases/coreweave-secures-1-1-billion..."),
    dict(company="Attentive Mobile Inc.", form_d_total_raised=529_046_678,
         round="Series D ($230M, Sep 2020) + Series E ($470M, Mar 2021)",
         lead_investors="Coatue (Series D)",
         other_investors="Tiger Global, Wellington Management, D1 Capital, Atomico, Sozo Ventures, Bain Capital Ventures, Sequoia, IVP, Eniac Ventures, NextView Ventures, High Alpha, Sapphire Ventures",
         source="attentive.com/blog/attentive-raises-230-million-series-d-investment-led-by-coatue"),
    dict(company="CRB Group, Inc.", form_d_total_raised=350_120_148,
         round="$50M common equity raise (Cross River Bank, parent entity)",
         lead_investors="existing Cross River investors + T. Rowe Price Investment Management",
         other_investors="earlier Series A (2016) led by Andreessen Horowitz, Battery Ventures, Ribbit Capital",
         source="roi-nj.com/2026/04/02/finance/cross-river-receives-50m..."),
    dict(company="Monad Labs, Inc.", form_d_total_raised=258_393_774,
         round="Series A ($225M, Apr 2024)",
         lead_investors="Paradigm ($150M of the round)",
         other_investors="Electric Capital, Coinbase Ventures, Castle Island Ventures, GSR Ventures, Greenoaks + notable angels",
         source="theblock.co/post/287257/monad-labs-raises-225-million-in-funding-round-led-by-paradigm"),
    dict(company="Rhodenergy An Environmental Co Corp", form_d_total_raised=250_000_000,
         round="not identified",
         lead_investors="not found",
         other_investors="no press coverage located under this name",
         source="n/a - no public press hit; may be too new / privately held"),
    dict(company="Legend Biotech Corp", form_d_total_raised=200_000_005,
         round="Post-IPO equity investment ($350M, May 2023); Legend Biotech is Nasdaq-listed (LEGN)",
         lead_investors="Hudson Bay Capital",
         other_investors="JJDC (Johnson & Johnson Innovation), Lilly Asia Ventures",
         source="bouncewatch.com/explore/startup/legend-biotech (aggregated)"),
    dict(company="Areteia Therapeutics, Inc.", form_d_total_raised=175_000_000,
         round="Series A ($350M total facility, closed Jul 2022)",
         lead_investors="Bain Capital Life Sciences",
         other_investors="Access Biotechnology, GV (Google Ventures), Arch Venture Partners, Saturn Partners, Sanofi, Population Health Partners, Maverick Capital",
         source="baincapital.com/news/knopp-biosciences-and-population-health-partners-create-areteia..."),
    dict(company="Apprentice FS, Inc.", form_d_total_raised=167_962_390,
         round="Series C ($62.71M, Mar 2023); ~$207M raised total",
         lead_investors="ICONIQ Capital / ICONIQ Growth",
         other_investors="Insight Partners, Alkeon Capital Management, Cerity Partners Ventures, Colorcon, Pacific Western Bank",
         source="roi-nj.com/2023/03/23/healthcare/jersey-city-based-apprentice-io-raises-65m..."),
    dict(company="UroGen Pharma Ltd.", form_d_total_raised=120_000_000,
         round="Private placement ($120M, Jul 2023); UroGen is Nasdaq-listed (URGN) -- this is a PIPE, not a VC round",
         lead_investors="RA Capital Management, Great Point Partners",
         other_investors="Acorn Bioventures, Monograph Capital, Horton Capital Partners",
         source="businesswire.com/news/home/20230727406893/en/UroGen-Announces-120-Million..."),
    dict(company="Reunion Neuroscience, Inc.", form_d_total_raised=119_850_000,
         round="Series A ($103M, May 2024)",
         lead_investors="MPM BioImpact, Novo Holdings (co-led)",
         other_investors="Arkin Bio Capital, Mitsui & Co. Global Investment, Plaisance Capital, FemHealth Ventures, Palo Santo",
         source="biopharmadive.com/news/reunion-neuroscience-psychedelic-biotech-series-a-investors"),
    dict(company="Antios Therapeutics Inc.", form_d_total_raised=106_999_993,
         round="Series B ($96M, Apr 2021) + Series B-1 ($75M, Nov 2021)",
         lead_investors="Soleus Capital (B); GordonMD Global Investments + EPIQ Capital Group (B-1)",
         other_investors="RA Capital, Adage Capital, Pontifax, Aisling Capital, Altium Capital, Amzak Health, Lumira Ventures, Domain Associates + others",
         source="globenewswire.com/news-release/2021/04/12/2208133/.../Antios-Therapeutics-Raises-96-Million..."),
])
investor_research

,company,form_d_total_raised,round,lead_investors,other_investors,source
0,"CoreWeave, Inc.",1158182236,"Series C ($1.1B, May 2024)",Coatue,"Magnetar, Altimeter Capital, Fidelity Manageme...",prnewswire.com/news-releases/coreweave-secures...
1,Attentive Mobile Inc.,529046678,"Series D ($230M, Sep 2020) + Series E ($470M, ...",Coatue (Series D),"Tiger Global, Wellington Management, D1 Capita...",attentive.com/blog/attentive-raises-230-millio...
2,"CRB Group, Inc.",350120148,"$50M common equity raise (Cross River Bank, pa...",existing Cross River investors + T. Rowe Price...,earlier Series A (2016) led by Andreessen Horo...,roi-nj.com/2026/04/02/finance/cross-river-rece...
3,"Monad Labs, Inc.",258393774,"Series A ($225M, Apr 2024)",Paradigm ($150M of the round),"Electric Capital, Coinbase Ventures, Castle Is...",theblock.co/post/287257/monad-labs-raises-225-...
4,Rhodenergy An Environmental Co Corp,250000000,not identified,not found,no press coverage located under this name,n/a - no public press hit; may be too new / pr...
5,Legend Biotech Corp,200000005,"Post-IPO equity investment ($350M, May 2023); ...",Hudson Bay Capital,"JJDC (Johnson & Johnson Innovation), Lilly Asi...",bouncewatch.com/explore/startup/legend-biotech...
6,"Areteia Therapeutics, Inc.",175000000,"Series A ($350M total facility, closed Jul 2022)",Bain Capital Life Sciences,"Access Biotechnology, GV (Google Ventures), Ar...",baincapital.com/news/knopp-biosciences-and-pop...
7,"Apprentice FS, Inc.",167962390,"Series C ($62.71M, Mar 2023); ~$207M raised total",ICONIQ Capital / ICONIQ Growth,"Insight Partners, Alkeon Capital Management, C...",roi-nj.com/2023/03/23/healthcare/jersey-city-b...
8,UroGen Pharma Ltd.,120000000,"Private placement ($120M, Jul 2023); UroGen is...","RA Capital Management, Great Point Partners","Acorn Bioventures, Monograph Capital, Horton C...",businesswire.com/news/home/20230727406893/en/U...
9,"Reunion Neuroscience, Inc.",119850000,"Series A ($103M, May 2024)","MPM BioImpact, Novo Holdings (co-led)","Arkin Bio Capital, Mitsui & Co. Global Investm...",biopharmadive.com/news/reunion-neuroscience-ps...


### Takeaways
- **10 of 15** top-raised companies had identifiable lead investors via press search; **1** (Rhodenergy) had no public coverage found; **4** were excluded as investment-vehicle entities rather than startups.
- No Y Combinator hits among any of these — consistent with YC skewing seed/early-stage, while this list is dominated by later-stage Series B/C/D and biotech rounds.
- Common threads: **Coatue** appears twice (CoreWeave, Attentive Mobile); heavy **biotech/pharma VC** presence (Bain Capital Life Sciences, RA Capital, MPM BioImpact, Novo Holdings) reflecting the `industry` breakdown from the EDA above (Biotechnology + Pharmaceuticals + Other Health Care are large chunks of the dataset).
- **UroGen** and **Legend Biotech** are both already-public companies raising via PIPE/follow-on, not early VC rounds — worth excluding from a "startup funding" narrative even though they show up as top Form D raises.
- This investor list is **not reproducible at scale** — it took ~10 manual searches for 15 companies. Extending to the full 778-company dataset isn't feasible without a paid data source (see earlier Crunchbase/PitchBook pricing check).

---
## Checking two more sources for investor identity: NJEDA and SEC Form C/C-AR

### SEC Form C / C-AR (Reg CF crowdfunding) — does NOT help
Confirmed via SEC's own Regulation Crowdfunding rules (Rule 201): Form C requires the
issuer's officers/directors and anyone owning 20%+ of voting stock, plus financial
statements — but **individual investor names and lead investors are never required**.
Same blind spot as Form D. Where Form C *would* help is the revenue-disclosure gap
(91% "Decline to Disclose" in our data) since Form C mandates real financial statements —
but that's a different question, and it only covers companies that used Reg CF (a
different, usually-smaller-raise exemption than Reg D), so it wouldn't extend our existing
778-company Form D list, just add different companies.

### NJEDA — actually helps, for a small subset
NJEDA's **NJ Innovation Evergreen Fund** publishes press releases each time the state
co-invests alongside a "Qualified Venture Firm," and these name both the company and the
VC firm. Worked through 4 of these releases (njeda.gov press archive) and
cross-referenced the named companies against our Form D dataset:

In [ ]:
njeda_press_releases = pd.DataFrame([
    dict(company="Antigenix Therapeutics, Inc.", njeda_amount=1_000_000,
         co_investing_firm="Pier 70 Ventures",
         source="njeda.gov/5-new-jersey-businesses-receive-funding-through-nj-innovation-evergreen-fund"),
    dict(company="PolyGone Systems, Inc.", njeda_amount=400_000,
         co_investing_firm="Tech Council Ventures",
         source="njeda.gov/5-new-jersey-businesses-receive-funding-through-nj-innovation-evergreen-fund"),
    dict(company="TranscendAP", njeda_amount=1_250_000,
         co_investing_firm="Rittenhouse Ventures & Tech Council Ventures",
         source="njeda.gov/5-new-jersey-businesses-receive-funding-through-nj-innovation-evergreen-fund"),
    dict(company="Lula, Inc.", njeda_amount=1_000_000,
         co_investing_firm="UP.Partners",
         source="njeda.gov/5-new-jersey-businesses-receive-funding-through-nj-innovation-evergreen-fund"),
    dict(company="Synchrony Medical", njeda_amount=1_000_000,
         co_investing_firm="Edge Medical Ventures",
         source="njeda.gov/5-new-jersey-businesses-receive-funding-through-nj-innovation-evergreen-fund"),
    dict(company="Nascent Materials, Inc.", njeda_amount=750_000,
         co_investing_firm="SOSV",
         source="njeda.gov/njeda-closes-on-two-new-nj-innovation-evergreen-fund-investments"),
    dict(company="Enquyst Technologies, Inc.", njeda_amount=3_000_000,
         co_investing_firm="Eckuity Capital",
         source="njeda.gov/njeda-closes-on-two-new-nj-innovation-evergreen-fund-investments"),
    dict(company="PolyGone Systems, Inc.", njeda_amount=1_250_000,
         co_investing_firm="FYRFLY Venture Partners",
         source="njeda.gov/3-new-jersey-businesses-secure-investments-through-njedas-evergreen-fund (separate/later round from the $400K Tech Council Ventures entry above)"),
    dict(company="JogoHealth, Inc.", njeda_amount=3_180_000,
         co_investing_firm="Atma Capital ($2M) & Creative Ventures Management ($1.18M)",
         source="njeda.gov/3-new-jersey-businesses-secure-investments-through-njedas-evergreen-fund"),
    dict(company="Hill Research", njeda_amount=1_750_000,
         co_investing_firm="Covenant Venture Capital",
         source="njeda.gov/3-new-jersey-businesses-secure-investments-through-njedas-evergreen-fund"),
    dict(company="Ricovr Healthcare Inc.", njeda_amount=400_000,
         co_investing_firm="Yaax Capital",
         source="njeda.gov/njeda-board-approves-nj-innovation-evergreen-fund-investment-into-princeton-based-life-sciences-company"),
])

# fuzzy match against our Form D company list
matches = companies[companies.company.str.contains(
    'Antigenix|PolyGone|Synchrony Medical|JogoHealth|Ricovr', case=False, na=False, regex=True
)][['company', 'city', 'industry', 'total_raised', 'total_investors', 'num_offerings', 'first_raise']]

print('NJEDA-named companies found in our Form D dataset:')
display(matches)
print()
print('Full NJEDA press-release table (includes companies NOT in our Form D dataset):')
njeda_press_releases

### Findings
All 9 distinct NJEDA-named companies, kept regardless of whether they matched our Form D
dataset — the ones that don't match still tell us who NJEDA/VCs are backing in NJ, just
not joinable to `total_raised` etc.:

| Company | Co-investing VC firm(s) (via NJEDA) | In our Form D dataset? |
|---|---|---|
| Antigenix Therapeutics, Inc. | Pier 70 Ventures | Yes |
| PolyGone Systems, Inc. | Tech Council Ventures (2024 round) + FYRFLY Venture Partners (later round) | Yes |
| Synchrony Medical | Edge Medical Ventures | Yes |
| JogoHealth, Inc. | Atma Capital + Creative Ventures Management | Yes |
| Ricovr Healthcare Inc. | Yaax Capital | Yes |
| TranscendAP | Rittenhouse Ventures & Tech Council Ventures | No — not found under this name |
| Lula, Inc. | UP.Partners | No — not found under this name |
| Nascent Materials, Inc. | SOSV | No — not found under this name |
| Hill Research | Covenant Venture Capital | No — not found under this name |
| Enquyst Technologies, Inc. | Eckuity Capital | No — not found under this name |

Unmatched likely means: too new to have filed Form D yet, filed under a slightly
different legal name, or raised via an instrument that didn't require a Form D (e.g. a
SAFE below the threshold, or grant/non-equity funding). Worth a manual EDGAR company-name
search for these 5 specifically if this matters for the writeup — a fuzzy `str.contains`
match isn't exhaustive.

**Scale reality check:** 10 companies came from working through 4 NJEDA press releases,
found via targeted search rather than a full archive crawl. NJEDA's press page likely has
dozens more going back to the 2022 program launch — continuing this exercise across the
full archive would probably add a few dozen more entries at most, and it's still fully
manual (no bulk API/download found for this data).

### Updated answer to "does this dataset answer the investor question"
Still **no** at the SEC-filing level (Form D and Form C both exclude investor identity by
design), but **yes, partially and manually**, by cross-referencing free secondary sources:
- Company press releases / funding announcements → worked for 10 of the top 15 by raise size
- NJEDA Evergreen Fund announcements → named investors for 10 companies total, 5 of which join back to our Form D dataset
20 named-investor data points total (15 that join to Form D + 5 NJEDA-only), out of 778
companies. That's the realistic ceiling for free manual research — full coverage would
require a paid tool (Crunchbase/PitchBook) or substantially more research time.